## Test automation - Logging and comparing results

This notebook is about showing how to test your LLM apps and send the test results to Azure Application Insights.

You can create dashboards using Azure data Explorer and even ship the logs to Microsoft Fabric for creating dashboards.

In [13]:
from dotenv import load_dotenv
import os

load_dotenv()
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_GPT4_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT_NAME")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
APPLICATIONINSIGHTS_CONNECTION_STRING = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")

In [14]:
from azure.ai.evaluation import AzureOpenAIModelConfiguration
from azure.identity import DefaultAzureCredential
from azure.ai.evaluation import (
    RelevanceEvaluator,
    CoherenceEvaluator,
    GroundednessEvaluator,
)
try:
    credential = DefaultAzureCredential()
    token = credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    print(ex)
    

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_deployment=AZURE_OPENAI_GPT4_DEPLOYMENT_NAME,
)

In [53]:
def test_coherence(query, response):
    coherence_evaluator = CoherenceEvaluator(model_config=model_config)
    score = coherence_evaluator(
        query=query, 
        response=response
    )
    score = 0 if score == None else score["gpt_coherence"]
    return score

def test_groundedness( response, context):
    groundedness_evaluator = GroundednessEvaluator(model_config=model_config)
    score = groundedness_evaluator(
        response=response,
        context=context,
    )
    score = 0 if score == None else score["gpt_groundedness"]
    return score

def test_relevance(query, response, context):
    relevance_eval = RelevanceEvaluator(model_config=model_config)
    score = relevance_eval(
        query=query, 
        response=response,
        context=context,
    )
    score = 0 if score == None else score["gpt_relevance"]
    return score

In [16]:
import os
import logging

from opentelemetry._logs import set_logger_provider
from opentelemetry.sdk._logs import (
    LoggerProvider,
    LoggingHandler,
)
from opentelemetry.sdk._logs.export import BatchLogRecordProcessor
from azure.monitor.opentelemetry.exporter import AzureMonitorLogExporter


def set_up_logging():
    logger_provider = LoggerProvider()
    set_logger_provider(logger_provider)

    exporter = AzureMonitorLogExporter(connection_string=APPLICATIONINSIGHTS_CONNECTION_STRING)
    logger_provider.add_log_record_processor(BatchLogRecordProcessor(exporter))

    # Attach LoggingHandler to namespaced logger
    handler = LoggingHandler()
    logger = logging.getLogger(__name__)
    logger.addHandler(handler)
    logger.setLevel(logging.NOTSET)
    return logger, logger_provider

In [17]:
# This must be done before any other telemetry calls
logger, logger_provider = set_up_logging()

In [54]:
import pandas as pd
import uuid
test_run_id = str(uuid.uuid4())

df = pd.read_csv("./data/test_data.csv")
for index, row in df.iterrows():
    query_id = row['query_id']
    test_type = row['test_type']
    query = row['query']
    prompt_name = 'test'
    answer = row['answer']
    context = row['context']
    if test_type == 'coherence':
        coherence = test_coherence(query, answer)
    elif test_type == 'groundedness':
        groundedness = test_groundedness(answer, context)
    elif test_type == 'relevance':
        relevance = test_relevance(query, answer, context)

    llm_properties = {
        'llm_version': AZURE_OPENAI_API_VERSION,
        'llm_model': AZURE_OPENAI_GPT4_DEPLOYMENT_NAME,
        'temperature': 0,
        'test_run_id': test_run_id,
        'query_id': query_id,
        'prompt_name': prompt_name,
        'coherence': coherence,
        'groundedness': groundedness,
        'relevance': relevance
    }
    logger.warning("Automatic testing", extra=llm_properties)

In [50]:
logger_provider.force_flush()

True